# Import thư viện

In [1]:
%%capture
!pip install -q transformers datasets evaluate accelerate

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import Mask2FormerImageProcessor, Mask2FormerForUniversalSegmentation
import numpy as np
import matplotlib.pyplot as plt
import cv2
import random
from tqdm.auto import tqdm
import warnings

warnings.filterwarnings("ignore")


In [2]:
# ==========================================
# CẤU HÌNH HỆ THỐNG 
# ==========================================
CFG = {
    "dataset_name": "EduardoPacheco/FoodSeg103",
    "model_ckpt": "facebook/mask2former-swin-tiny-ade-semantic",
    "num_classes": 104,
    "image_size": 256,   
    "batch_size": 8,     
    
    
    "learning_rate": 1e-4, 
    "min_lr": 1e-6,        
    "lr_drop_factor": 0.5, 
    "lr_patience": 1,     
    "early_stop_patience": 4, 
    "epochs": 50,          
    
    "ignore_index": 255,
    "save_path": "mask2former_best_model.pth",
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu")
}

## 1. Chuẩn bị & Tiền xử lý Dữ liệu (Data Pipeline)
Khác với mạng CNN truyền thống, **Mask2Former** yêu cầu bộ dữ liệu nhãn (Ground Truth) phải được băm nhỏ thành các mặt nạ nhị phân (binary masks) cho từng nhãn lớp xuất hiện trong ảnh. `Mask2FormerImageProcessor` sẽ tự động thực hiện phép biến đổi phức tạp này.

In [3]:
print(" Đang tải dữ liệu FoodSeg103...")
dataset = load_dataset(CFG["dataset_name"])

# Khởi tạo bộ xử lý của Mask2Former
processor = Mask2FormerImageProcessor(
    ignore_index=CFG["ignore_index"], 
    reduce_labels=False, 
    do_resize=True, 
    size={"height": CFG["image_size"], "width": CFG["image_size"]}
)

# Hàm transform băm mask
def train_transforms(example_batch):
    images = [img.convert("RGB") for img in example_batch['image']]
    labels = [np.array(lbl) for lbl in example_batch['label']]
    # Processor tự động tạo mask_labels và class_labels
    inputs = processor(images, segmentation_maps=labels, return_tensors="pt")
    return inputs

train_dataset = dataset["train"].with_transform(train_transforms)
val_dataset = dataset["validation"].with_transform(train_transforms)

# Hàm gộp batch bắt buộc cho Mask2Former
def collate_fn(batch):
    pixel_values = torch.stack([example["pixel_values"] for example in batch])
    pixel_mask = torch.stack([example["pixel_mask"] for example in batch])
    mask_labels = [example["mask_labels"] for example in batch]
    class_labels = [example["class_labels"] for example in batch]
    return {
        "pixel_values": pixel_values,
        "pixel_mask": pixel_mask,
        "mask_labels": mask_labels,
        "class_labels": class_labels
    }

train_dataloader = DataLoader(train_dataset, batch_size=CFG["batch_size"], shuffle=True, drop_last=True, collate_fn=collate_fn)
val_dataloader = DataLoader(val_dataset, batch_size=CFG["batch_size"], shuffle=False, collate_fn=collate_fn)
print(" Tải dữ liệu hoàn tất! Đã sẵn sàng DataLoader.")

 Đang tải dữ liệu FoodSeg103...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00003.parquet:   0%|          | 0.00/351M [00:00<?, ?B/s]

data/train-00001-of-00003.parquet:   0%|          | 0.00/357M [00:00<?, ?B/s]

data/train-00002-of-00003.parquet:   0%|          | 0.00/431M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/115M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4983 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2135 [00:00<?, ? examples/s]

 Tải dữ liệu hoàn tất! Đã sẵn sàng DataLoader.


## 2. Khởi tạo Mô hình & Vòng lặp Huấn luyện (Training Loop)
Tải trọng số pre-trained của Mask2Former. Hàm Loss của Mask2Former là sự kết hợp phức tạp giữa **Cross-Entropy Loss**, **Dice Loss** và **Bipartite Matching Loss**, được mô hình tự động tính toán trong quá trình Forward Pass.

In [4]:
# Tải mô hình Mask2Former
model = Mask2FormerForUniversalSegmentation.from_pretrained(
    CFG["model_ckpt"],
    id2label={i: f"Class_{i}" for i in range(CFG["num_classes"])},
    ignore_mismatched_sizes=True
).to(CFG["device"])

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["learning_rate"])

def get_current_lr(optimizer):
    for param_group in optimizer.param_groups:
        return param_group['lr']

def compute_iou_quick(preds, labels, num_classes, ignore_index=255):
    valid_mask = (labels != ignore_index)
    preds = preds[valid_mask]
    labels = labels[valid_mask]
    inter = torch.bincount(labels[preds == labels], minlength=num_classes)
    pred_counts = torch.bincount(preds, minlength=num_classes)
    label_counts = torch.bincount(labels, minlength=num_classes)
    union = pred_counts + label_counts - inter
    return inter, union

# --- BẮT ĐẦU HUẤN LUYỆN ---
best_miou = 0.0
patience_counter = 0

for epoch in range(CFG["epochs"]):
    model.train()
    running_loss = 0.0
    current_lr = get_current_lr(optimizer)
    
    print(f"\n[Epoch {epoch+1}/{CFG['epochs']}] Đang huấn luyện Mask2Former... (LR: {current_lr:.6f})")
    
    for batch in tqdm(train_dataloader, desc="Training"):
        optimizer.zero_grad()
        
        pixel_values = batch["pixel_values"].to(CFG["device"])
        mask_labels = [labels.to(CFG["device"]) for labels in batch["mask_labels"]]
        class_labels = [labels.to(CFG["device"]) for labels in batch["class_labels"]]
        
        outputs = model(pixel_values=pixel_values, mask_labels=mask_labels, class_labels=class_labels)
        
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        
    print(f"-> Train Loss: {running_loss / len(train_dataloader):.4f}")
    
    # --- ĐÁNH GIÁ (Fix lỗi kích thước) ---
    model.eval()
    total_inter = torch.zeros(CFG["num_classes"]).to(CFG["device"])
    total_union = torch.zeros(CFG["num_classes"]).to(CFG["device"])
    
    with torch.no_grad():
        for batch_idx, batch in enumerate(tqdm(val_dataloader, desc="Evaluating")):
            pixel_values = batch["pixel_values"].to(CFG["device"])
            outputs = model(pixel_values=pixel_values)
            
            # 1. Trích xuất kích thước GỐC của từng tấm ảnh trong mẻ này
            target_sizes = []
            true_labels = []
            for idx in range(len(pixel_values)):
                dataset_idx = batch_idx * CFG["batch_size"] + idx
                raw_label = np.array(dataset['validation'][dataset_idx]['label'])
                target_sizes.append(raw_label.shape) # (Chiều cao, Chiều rộng gốc)
                true_labels.append(raw_label)
            
            # 2. Ép mô hình nhả mask bằng ĐÚNG kích thước gốc đó
            pred_maps = processor.post_process_semantic_segmentation(outputs, target_sizes=target_sizes)
            
            # 3. Chấm điểm cực chuẩn
            for idx, pred_map in enumerate(pred_maps):
                true_label = torch.tensor(true_labels[idx], device=CFG["device"])
                inter, union = compute_iou_quick(pred_map, true_label, CFG["num_classes"])
                total_inter += inter
                total_union += union
                
    ious = total_inter / (total_union + 1e-6)
    current_miou = ious.mean().item() * 100
    print(f"-> Test mIoU: {current_miou:.2f}%")
    
    # ==========================================
    # CƠ CHẾ EARLY STOPPING 
    # ==========================================
    if current_miou > best_miou:
        best_miou = current_miou
        patience_counter = 0
        torch.save(model.state_dict(), CFG["save_path"])
        print(f"🌟 Đỉnh mới! Đã lưu trọng số tốt nhất (mIoU: {best_miou:.2f}%)")
    else:
        patience_counter += 1
        print(f"⚠️ Điểm chững lại. Mức cảnh báo: {patience_counter}/{CFG['early_stop_patience']}")
        
        if patience_counter >= CFG["lr_patience"]:
            new_lr = max(current_lr * CFG["lr_drop_factor"], CFG["min_lr"])
            if new_lr < current_lr:
                for param_group in optimizer.param_groups:
                    param_group['lr'] = new_lr
                print(f"Đã giảm Learning Rate xuống: {new_lr:.6f}")
        
        if patience_counter >= CFG["early_stop_patience"]:
            print("\n🛑 KÍCH HOẠT DỪNG SỚM!")
            break

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/190M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/566 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/190M [00:00<?, ?B/s]


[Epoch 1/50] Đang huấn luyện Mask2Former... (LR: 0.000100)


...

-> Test mIoU: 33.93%
⚠️ Điểm chững lại. Mức cảnh báo: 3/4

[Epoch 23/50] Đang huấn luyện Mask2Former... (LR: 0.000001)


Training:   0%|          | 0/622 [00:00<?, ?it/s]

-> Train Loss: 9.0196


Evaluating:   0%|          | 0/267 [00:00<?, ?it/s]

-> Test mIoU: 33.39%
⚠️ Điểm chững lại. Mức cảnh báo: 4/4

🛑 KÍCH HOẠT DỪNG SỚM!


## 3. Đánh Giá Độ Đo Đạt Chuẩn Báo Cáo Khoa Học (mIoU, mAcc, aAcc)
Sau khi kết thúc quá trình huấn luyện, tải lại bộ trọng số tốt nhất để tính toán 3 độ đo cốt lõi của bài toán Semantic Segmentation, mô phỏng chính xác phương pháp đánh giá của tác giả bộ dữ liệu FoodSeg103.

In [5]:
print("⚙️ Đang nạp lại bộ não Mask2Former xuất sắc nhất...")
best_model = Mask2FormerForUniversalSegmentation.from_pretrained(
    CFG["model_ckpt"],
    id2label={i: f"Class_{i}" for i in range(CFG["num_classes"])},
    ignore_mismatched_sizes=True
).to(CFG["device"])

state_dict = torch.load(CFG["save_path"], map_location=CFG["device"])
best_model.load_state_dict(state_dict)
best_model.eval()

total_inter = torch.zeros(CFG["num_classes"]).to(CFG["device"])
total_union = torch.zeros(CFG["num_classes"]).to(CFG["device"])
total_label_counts = torch.zeros(CFG["num_classes"]).to(CFG["device"])
total_correct_pixels = 0
total_valid_pixels = 0

print("🔍 Đang chấm điểm chi tiết trên toàn bộ tập Validation...")
with torch.no_grad():
    for batch_idx, batch in enumerate(tqdm(val_dataloader, desc="Scoring")):
        pixel_values = batch["pixel_values"].to(CFG["device"])
        outputs = best_model(pixel_values=pixel_values)
        
        # 1. Trích xuất kích thước GỐC 
        target_sizes = []
        true_labels = []
        for idx in range(len(pixel_values)):
            dataset_idx = batch_idx * CFG["batch_size"] + idx
            raw_label = np.array(dataset['validation'][dataset_idx]['label'])
            target_sizes.append(raw_label.shape)
            true_labels.append(raw_label)
        
        # 2. Phóng to ảnh dự đoán khớp y xì ảnh gốc
        pred_maps = processor.post_process_semantic_segmentation(outputs, target_sizes=target_sizes)
        
        # 3. So khớp từng Pixel
        for idx, pred_map in enumerate(pred_maps):
            true_label = torch.tensor(true_labels[idx], device=CFG["device"])
            
            valid_mask = (true_label != CFG["ignore_index"])
            valid_preds = pred_map[valid_mask]
            valid_labels = true_label[valid_mask]
            
            inter = torch.bincount(valid_labels[valid_preds == valid_labels], minlength=CFG["num_classes"])
            pred_counts = torch.bincount(valid_preds, minlength=CFG["num_classes"])
            label_counts = torch.bincount(valid_labels, minlength=CFG["num_classes"])
            union = pred_counts + label_counts - inter
            
            total_inter += inter
            total_union += union
            total_label_counts += label_counts
            total_correct_pixels += (valid_preds == valid_labels).sum().item()
            total_valid_pixels += valid_mask.sum().item()

# TÍNH TOÁN FINAL METRICS
ious = total_inter / (total_union + 1e-6)
final_miou = ious.mean().item() * 100

class_accs = total_inter / (total_label_counts + 1e-6)
final_macc = class_accs.mean().item() * 100

pixel_acc = (total_correct_pixels / total_valid_pixels) * 100

print("\n" + "="*45)
print(" KẾT QUẢ ĐÁNH GIÁ MASK2FORMER (FOODSEG103) ")
print("="*45)
print(f" 1. mIoU (Độ khớp vùng trung bình): {final_miou:.2f}%")
print(f" 2. mAcc (Độ chính xác từng món):   {final_macc:.2f}%")
print(f" 3. aAcc (Độ chính xác điểm ảnh):   {pixel_acc:.2f}%")
print("="*45)

⚙️ Đang nạp lại bộ não Mask2Former xuất sắc nhất...


Loading weights:   0%|          | 0/566 [00:00<?, ?it/s]

🔍 Đang chấm điểm chi tiết trên toàn bộ tập Validation...


Scoring:   0%|          | 0/267 [00:00<?, ?it/s]


 KẾT QUẢ ĐÁNH GIÁ MASK2FORMER (FOODSEG103) 
 1. mIoU (Độ khớp vùng trung bình): 34.30%
 2. mAcc (Độ chính xác từng món):   47.24%
 3. aAcc (Độ chính xác điểm ảnh):   80.26%


## 4. Trực Quan Hóa Bảng Kết Quả (Qualitative Results)
Xuất 5 mẫu ảnh ngẫu nhiên từ tập Validation để đối chiếu: Ảnh Gốc - Nhãn Chuẩn (Ground Truth) - Kết quả suy luận của Mask2Former. Model sẽ tự động kéo giãn kích thước mask dự đoán về lại đúng kích thước ảnh gốc để đắp lớp màu overlay.

In [6]:
print(" Đang xuất bảng vẽ 5 ảnh so sánh Trực quan...")

# Lấy bản raw chưa qua transform để có ảnh gốc
raw_val = dataset['validation'].with_transform(None)
sample_indices = random.sample(range(len(raw_val)), 5)
samples = [raw_val[i] for i in sample_indices]

fig, axes = plt.subplots(5, 3, figsize=(18, 25))
fig.suptitle("So Sánh Hiệu Quả Phân Vùng đồ ăn: Ground Truth vs Mask2Former", 
             fontsize=22, fontweight='bold', y=0.98)

best_model.eval()
with torch.no_grad():
    for i, sample in enumerate(samples):
        # 1. Ảnh gốc
        raw_img = sample['image'].convert("RGB")
        true_mask = np.array(sample['label'])
        
        # 2. Xử lý qua model
        inputs = processor(images=raw_img, return_tensors="pt")
        pixel_values = inputs["pixel_values"].to(CFG["device"])
        
        # Yêu cầu Mask2Former nhả mask ra đúng kích thước ảnh gốc
        outputs = best_model(pixel_values=pixel_values)
        target_size = [(raw_img.height, raw_img.width)] 
        pred_map = processor.post_process_semantic_segmentation(outputs, target_sizes=target_size)[0]
        pred_mask = pred_map.cpu().numpy().astype(np.uint8)
        
        # --- CỘT 1: ẢNH GỐC ---
        axes[i, 0].imshow(raw_img)
        if i == 0: axes[i, 0].set_title("1. Ảnh Gốc (Input)", fontsize=16, fontweight='bold')
        axes[i, 0].axis('off')
        
        # --- CỘT 2: GROUND TRUTH ---
        axes[i, 1].imshow(raw_img)
        masked_true = np.ma.masked_where((true_mask == 0) | (true_mask == CFG["ignore_index"]), true_mask)
        axes[i, 1].imshow(masked_true, cmap='nipy_spectral', alpha=0.65)
        if i == 0: axes[i, 1].set_title("2. Nhãn Chuẩn (Ground Truth)", fontsize=16, fontweight='bold')
        axes[i, 1].axis('off')
        
        # --- CỘT 3: DỰ ĐOÁN ---
        axes[i, 2].imshow(raw_img)
        masked_pred = np.ma.masked_where((pred_mask == 0), pred_mask)
        axes[i, 2].imshow(masked_pred, cmap='nipy_spectral', alpha=0.65)
        if i == 0: axes[i, 2].set_title("3. Mask2Former Prediction", fontsize=16, fontweight='bold', color='darkred')
        axes[i, 2].axis('off')

plt.tight_layout()
plt.subplots_adjust(top=0.95)
plt.show()

 Đang xuất bảng vẽ 5 ảnh so sánh Trực quan...


TypeError: 'NoneType' object is not callable